### Import libraries

In [1]:
import os
import pickle
from methyldl.deconvolution.xgbdeconvolver import *
from tqdm import tqdm
from collections import defaultdict
import torch.nn as nn
from copy import deepcopy
from methyldl.deconvolution.deep_deconvolvers.training import train_matrix_deconvolver
import os.path as Path
from torch.utils.data import DataLoader, TensorDataset
from methyldl.deconvolution.least_squares_deconvolvers import (
    NNLSDeconvolver,
    PSLSDeconvolver,
)
from methyldl.deconvolution.evaluation import compute_deconvolution_metrics
from methyldl.deconvolution.linear_calibrator import LinearCalibrator

### Configuring target model and extracting computed average scores

In [38]:
from edautils import *

reads_data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsForRRBSsplits_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_d041/"
dmr_label_column = "dmr_ctype_label"
mincpg_pointer = reads_data_path.find("mincpg_")
# classifier_model_path = f"../Tutorials/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_{dmr_label_column}_{reads_data_path[mincpg_pointer:]}"
classifier_model_path = "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Experiments/RRBS/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_d041_postfiltered_min_length_50_hard_labels_minibatch_balanced/pseudobulk"
mincpg = int(reads_data_path[mincpg_pointer + 7 : mincpg_pointer + 8])
features_encoding = "extracted_numpy"
n_cell_types = num_dmr_groups = 39
n_pred_classes = 39 if "soft_labels" in classifier_model_path else 40

In [43]:
features_file = [
    x for x in os.listdir(classifier_model_path) if "features_cutoff" in x
][0]

In [40]:
df = np.load(Path.join(classifier_model_path, features_file))

In [42]:
# features_test = df["features_test"]
features_train = df["features_train"]
features_valid = df["features_valid"]
target_proportions = df["proportions"]

### Defining deconvolvers

In [44]:
n_features = len(features_test[0])

In [45]:
swn = nn.Sequential(
    nn.Linear(n_features, 1024),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(1024, 39),
    nn.Softmax(dim=-1),
)

mlp = nn.Sequential(
    nn.Linear(n_features, 512),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(512, 256),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(256, n_features),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(n_features, 39),
    nn.Softmax(dim=-1),
)

xgb_config = XGBDeconvolverConfig(
    n_estimators=500,
    max_depth=15,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=1,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=100,
    random_state=42,
)

xgb = XGBoostDeconvolver(
    config=xgb_config,
    output_transform="clip_normalize",
    n_dmr_groups=num_dmr_groups,
    n_pred_classes=n_pred_classes,
    n_cell_types=n_cell_types,
    with_reject_features=False,
    process_inputs=False,
)

nnls = NNLSDeconvolver()
psls = PSLSDeconvolver()

### Loading deconvolvers weights

In [46]:
device = "cuda"

In [47]:
swn.to(device=device)
swn.load_state_dict(
    torch.load(
        Path.join(classifier_model_path, "swn_best_deconvolver.pt"), weights_only=True
    )
)
swn.eval()
mlp.to(device=device)
mlp.load_state_dict(
    torch.load(
        Path.join(classifier_model_path, "mlp_best_deconvolver.pt"), weights_only=True
    )
)
mlp.eval()
xgb = xgb.load(Path.join(classifier_model_path, "xgb_deconvolver.joblib"))
nnls = nnls.load(Path.join(classifier_model_path, "nnls_deconvolver.joblib"))
psls = psls.load(Path.join(classifier_model_path, "psls_deconvolver.joblib"))

### Generating predictions

In [48]:
results = defaultdict(tuple)
model_tripples = [
    (swn, "swn", "nn"),
    (mlp, "mlp", "nn"),
    (xgb, "xgb", "xgb"),
    (nnls, "nnls", "ls"),
    (psls, "psls", "ls"),
]

In [49]:
def infer_multiple_deconvolvers(features, target_proportions, model_tripples):
    test_loader = DataLoader(
        TensorDataset(
            torch.FloatTensor(features), torch.FloatTensor(target_proportions)
        ),
        batch_size=2000,
    )
    results = defaultdict(tuple)
    for model, model_name, model_type in model_tripples:
        all_preds = []
        all_targets = target_proportions
        if model_type == "nn":
            all_targets = []
            with torch.no_grad():
                for X, y in test_loader:
                    X, y = X.to(device), y.to(device)
                    pred = model(X)
                    all_preds.append(pred.cpu())
                    all_targets.append(y.cpu())
            # Compute all metrics
            all_preds = torch.cat(all_preds, dim=0).numpy()
            all_targets = torch.cat(all_targets, dim=0)
            all_targets = all_targets.numpy()
        elif model_type == "xgb":
            all_preds = model._predict_raw(features)
            all_preds = model._transform_output(all_preds)
        elif model_type == "ls":
            if "nnls" in model_name:
                all_preds, _, _ = model.predict(features, n_workers=1)
            elif "psls" in model_name:
                all_preds = model.predict(features, n_workers=2)
        else:
            pass
        metrics = compute_deconvolution_metrics(all_preds, all_targets)
        all_preds = np.round(all_preds, 4)

        results[model_name] = (all_targets, all_preds, metrics)

    return results

In [13]:
results_test = infer_multiple_deconvolvers(
    features=features_test,
    target_proportions=target_proportions,
    model_tripples=model_tripples,
)

Predicting with PSLS in parallel: 100%|██████████| 1000/1000 [02:00<00:00,  8.29it/s]


In [14]:
import pandas as pd


def results_to_dataframe(results: dict, class_names: list = None) -> pd.DataFrame:
    """
    Convert the results dict into a DataFrame matching the supplementary table columns.

    Expected results structure:
        results[model_name] = (all_targets, all_preds, metrics_dict)

    model_name convention assumed (adjust parsing as needed):
        e.g. "hard_rej__dsimir__dirichlet__xgb__linear"
              labeling__classifier__clf_calib__deconvolver__dec_calib
    """
    rows = []
    for model_name, (targets, preds, metrics) in results.items():
        # --- parse model_name into table keys ---
        parts = model_name.split("__")
        if len(parts) == 5:
            labeling, classifier, clf_calib, deconvolver, dec_calib = parts
        elif len(parts) == 3:
            # UXM case: e.g. "uxm__uxm__linear"
            labeling, deconvolver, dec_calib = parts
            classifier = "---"
            clf_calib = "---"
        else:
            # fallback: store raw name, fill manually
            labeling = ""
            classifier = ""
            clf_calib = ""
            deconvolver = model_name
            dec_calib = ""

        worst_class = metrics["worst_class_name"]
        if class_names is not None and isinstance(worst_class, int):
            worst_class = class_names[worst_class]

        rows.append(
            {
                # "Labeling": labeling,
                # "Classifier": classifier,
                # "Clf. Calib.": clf_calib,
                "Deconvolver": deconvolver,
                # "Dec. Calib.": dec_calib,
                "R2": (
                    1.0 - metrics["mse"] / targets.var()
                    if hasattr(targets, "var")
                    else None
                ),
                "LoA": f"[{metrics['loa_lower']:.4f}, {metrics['loa_upper']:.4f}]",
                "LoA width": round(metrics["loa_width"], 4),
                "LoA (worst)": f"[{metrics['worst_class_loa_lower']:.4f}, {metrics['worst_class_loa_upper']:.4f}]",
                "LoA width (worst)": round(metrics["worst_class_loa_width"], 4),
                "Worst class": worst_class,
                "MSE": round(metrics["mse"], 6),
                "MAE": round(metrics["mae"], 6),
                "KL": round(metrics["kl"], 6),
                "Cosine Sim": round(metrics["cosine_sim"], 6),
            }
        )

    df = pd.DataFrame(rows)

    # Sort to match table grouping order
    df = df.sort_values(by=["Deconvolver"]).reset_index(drop=True)

    return df

In [ ]:
results_to_dataframe(results_test)

{'Deconvolver': {0: 'mlp', 1: 'nnls', 2: 'psls', 3: 'swn', 4: 'xgb'},
 'R2': {0: 0.9608850479125977,
  1: 0.9772423987409544,
  2: 0.9771817238522059,
  3: 0.9783931970596313,
  4: 0.942941138765889},
 'LoA': {0: '[-0.0353, 0.0353]',
  1: '[-0.0269, 0.0269]',
  2: '[-0.0269, 0.0269]',
  3: '[-0.0262, 0.0262]',
  4: '[-0.0426, 0.0426]'},
 'LoA width': {0: 0.0706, 1: 0.0538, 2: 0.0539, 3: 0.0524, 4: 0.0852},
 'LoA (worst)': {0: '[-0.0946, 0.1378]',
  1: '[-0.0814, 0.0821]',
  2: '[-0.0802, 0.0819]',
  3: '[-0.0643, 0.0801]',
  4: '[-0.1048, 0.1161]'},
 'LoA width (worst)': {0: 0.2325, 1: 0.1635, 2: 0.1621, 3: 0.1444, 4: 0.2209},
 'Worst class': {0: 11, 1: 11, 2: 11, 3: 11, 4: 11},
 'MSE': {0: 0.000324, 1: 0.000188, 2: 0.000189, 3: 0.000179, 4: 0.000473},
 'MAE': {0: 0.004854, 1: 0.003881, 2: 0.003917, 3: 0.00364, 4: 0.005439},
 'KL': {0: 0.103988, 1: 0.101679, 2: 0.096849, 3: 0.067234, 4: 0.115015},
 'Cosine Sim': {0: 0.977477,
  1: 0.989655,
  2: 0.989637,
  3: 0.988771,
  4: 0.984705}}

### Infering linear callibrators

In [16]:
def apply_fitted_callibration(results_test):
    results = defaultdict(tuple)
    for model_name in results_test.keys():
        test_preds = results_test[model_name][0]
        test_target = results_test[model_name][1]

        calibrator = LinearCalibrator()
        calibrator.load_calibration_parameters(
            Path.join(classifier_model_path, f"{model_name}_linear_calibrator.npz")
        )
        calibrated_test_pred, _ = calibrator.predict(test_preds)

        metrics = compute_deconvolution_metrics(calibrated_test_pred, test_target)
        calibrated_test_pred = np.round(calibrated_test_pred, 4)

        results[model_name] = (test_target, calibrated_test_pred, metrics)

    return results

In [17]:
results_calibrated = apply_fitted_callibration(results_test)

In [18]:
results_to_dataframe(results_calibrated)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.951724,"[-0.0381, 0.0381]",0.0762,"[-0.1424, 0.0931]",0.2356,11,0.000378,0.005381,0.635776,0.973006
1,nnls,0.962818,"[-0.0325, 0.0325]",0.0650,"[-0.1033, 0.1178]",0.2211,11,0.000275,0.004672,0.851841,0.982444
2,psls,0.963357,"[-0.0322, 0.0322]",0.0645,"[-0.1011, 0.1136]",0.2148,11,0.000271,0.004681,0.862389,0.982832
3,swn,0.964706,"[-0.0325, 0.0325]",0.0650,"[-0.0912, 0.0698]",0.1610,11,0.000275,0.004589,0.386419,0.981178
4,xgb,0.918522,"[-0.0449, 0.0449]",0.0898,"[-0.1149, 0.0989]",0.2139,11,0.000525,0.006066,1.235024,0.980817


### Fitting linear callibrators

In [50]:
results_valid = infer_multiple_deconvolvers(
    features=features_valid,
    target_proportions=target_proportions,
    model_tripples=model_tripples,
)

Predicting with PSLS in parallel: 100%|██████████| 1000/1000 [02:07<00:00,  7.85it/s]


In [51]:
results_to_dataframe(results_valid)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.939680,"[-0.0438, 0.0438]",0.0876,"[-0.1394, 0.2071]",0.3465,11,0.000500,0.004581,0.157411,0.972066
1,nnls,0.981607,"[-0.0242, 0.0242]",0.0484,"[-0.0651, 0.0750]",0.1401,11,0.000152,0.003397,0.084321,0.991446
2,psls,0.981478,"[-0.0243, 0.0243]",0.0485,"[-0.0652, 0.0750]",0.1402,11,0.000153,0.003433,0.080976,0.991384
3,swn,0.968190,"[-0.0318, 0.0318]",0.0636,"[-0.0940, 0.1435]",0.2375,11,0.000263,0.003763,0.072744,0.982437
4,xgb,0.953660,"[-0.0384, 0.0384]",0.0768,"[-0.0940, 0.1043]",0.1983,11,0.000384,0.004720,0.095110,0.988157


In [52]:
def apply_callibration(results_valid, results_test, save_calibrators=False):
    results = defaultdict(tuple)
    for model_name in results_valid.keys():
        valid_preds = results_valid[model_name][0]
        val_target = results_valid[model_name][1]

        test_preds = results_test[model_name][0]
        test_target = results_test[model_name][1]

        calibrator = LinearCalibrator()
        calibrator.fit(valid_preds, val_target)
        calibrated_test_pred, _ = calibrator.predict(test_preds)

        metrics = compute_deconvolution_metrics(calibrated_test_pred, test_target)
        calibrated_test_pred = np.round(calibrated_test_pred, 4)

        results[model_name] = (test_target, calibrated_test_pred, metrics)

        if save_calibrators:
            calibrator.save_calibration_parameters(
                f"{classifier_model_path}/{model_name}_linear_calibrator.npz"
            )

    return results

In [53]:
calibrated_test_results = apply_callibration(
    results_valid, results_valid, save_calibrators=True
)

In [54]:
results_to_dataframe(calibrated_test_results)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.946588,"[-0.0401, 0.0401]",0.0802,"[-0.1693, 0.1674]",0.3367,11,0.000419,0.005200,0.160215,0.975688
1,nnls,0.987656,"[-0.0189, 0.0189]",0.0378,"[-0.0627, 0.0644]",0.1272,11,0.000093,0.003427,0.107981,0.994248
2,psls,0.987538,"[-0.0190, 0.0190]",0.0379,"[-0.0629, 0.0645]",0.1274,11,0.000094,0.003468,0.111211,0.994180
3,swn,0.974248,"[-0.0277, 0.0278]",0.0555,"[-0.1181, 0.1156]",0.2337,11,0.000200,0.004022,0.121055,0.986137
4,xgb,0.963670,"[-0.0304, 0.0304]",0.0609,"[-0.0867, 0.0891]",0.1758,11,0.000241,0.005986,0.154423,0.988499


### Latex entry generation

In [28]:
from collections import OrderedDict


def build_supp_table(
    entries,
    caption="Supplementary: Full results.",
    label="tab:supp_results",
):
    """
    Build a LaTeX supplementary table from a flat list of (classifier, calibration, data)
    tuples, where `data` is a dict-of-dicts in the shape returned by
    `pd.DataFrame.to_dict()` — i.e. {column_name: {row_idx: value, ...}, ...}.

    Each (classifier, calibration) pair contributes one row per Deconvolver in `data`.
    Rows are grouped by classifier via \\multirow; within a classifier, rows are grouped
    by Deconvolver so that calibration variants sit under the same Deconvolver header
    with \\cmidrule separators between different Deconvolvers.

    Columns emitted (with scaling applied to displayed values):
        Classifier | Deconvolver | Dec. Calib.
        | R^2 (×10⁻²) | LoA (×10⁻²) | LoA worst-class (×10⁻²)
        | MAE (×10⁻³)  | MSE (×10⁻⁴) | KL (×10⁻²)

    Raw input values are multiplied by the appropriate factor so the displayed
    numbers match the column headers.

    No automatic highlighting is applied.

    Parameters
    ----------
    entries : list[tuple[str, str, dict]]
        Each tuple is (classifier_name, calibration_name, data_dict).
        `data_dict` must contain the keys:
            'Deconvolver', 'R2', 'LoA', 'LoA (worst)', 'MAE', 'MSE', 'KL'
        R2 is expected as a fraction (e.g. 0.9787), not a percentage.
        LoA and LoA (worst) are strings like '[-0.0260, 0.0260]'.
        MAE, MSE, KL are floats.
    caption : str
    label : str

    Returns
    -------
    str : The full LaTeX table as a string.
    """
    # Deconvolver display names (upper-case) — input may use any casing
    decon_display = {
        "xgb": "XGB",
        "mlp": "MLP",
        "swn": "SWN",
        "nnls": "NNLS",
        "psls": "PSLS",
    }

    def scale_loa(loa_str, factor=100):
        """Scale a LoA string like '[-0.0260, 0.0260]' by `factor`."""
        stripped = loa_str.strip().strip("[]")
        parts = [float(x.strip()) for x in stripped.split(",")]
        scaled = [x * factor for x in parts]
        return f"[{scaled[0]:.2f}, {scaled[1]:.2f}]"

    # 1) Group entries by classifier, preserving input order
    by_classifier = OrderedDict()
    for clf, calib, data in entries:
        by_classifier.setdefault(clf, []).append((calib, data))

    # 2) For each classifier, reorganize into {deconvolver: [(calib, row_values), ...]}
    def format_row_values(data, row_idx):
        r2 = data["R2"][row_idx] * 100  # ×10⁻² → display as e.g. 97.87
        mae = data["MAE"][row_idx] * 1000  # ×10⁻³ → display as e.g. 3.585
        mse = data["MSE"][row_idx] * 10000  # ×10⁻⁴ → display as e.g. 1.76
        kl = data["KL"][row_idx] * 100  # ×10⁻² → display as e.g. 9.33
        loa = scale_loa(data["LoA"][row_idx])  # ×10⁻² → e.g. [-2.60, 2.60]
        loa_worst = scale_loa(data["LoA (worst)"][row_idx])
        return {
            "R2": f"{r2:.2f}",
            "LoA": loa,
            "LoA (worst)": loa_worst,
            "MAE": f"{mae:.3f}",
            "MSE": f"{mse:.2f}",
            "KL": f"{kl:.2f}",
        }

    def collect_by_decon(calib_data_list):
        """Return OrderedDict: decon_name -> list of (calib_name, formatted_row_dict)."""
        by_decon = OrderedDict()
        for calib, data in calib_data_list:
            for row_idx, decon_raw in data["Deconvolver"].items():
                decon = decon_display.get(decon_raw.lower(), decon_raw.upper())
                by_decon.setdefault(decon, []).append(
                    (calib, format_row_values(data, row_idx))
                )
        return by_decon

    # 3) Emit LaTeX
    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"  \centering")
    lines.append(rf"  \caption{{{caption}}}")
    lines.append(rf"  \label{{{label}}}")
    lines.append(r"  {\footnotesize")
    lines.append(r"    \resizebox{\textwidth}{!}{%")
    lines.append(r"      \begin{tabular}{l cc cccc cc}")
    lines.append(r"        \toprule")
    # Two-row header with units on the second row
    lines.append(
        r"        \multirow{2}{*}{\textbf{Classifier}} & \multirow{2}{*}{\textbf{Deconvolver}}"
        r" & \multirow{2}{*}{\textbf{Dec.\ Calib.}}"
        r" & \textbf{$R^2$} & \textbf{LoA} & \textbf{LoA worst-class} & \textbf{MAE}"
        r" & \textbf{MSE} & \textbf{KL} \\"
    )
    lines.append(
        r"        & & "
        r" & ($\times 10^{-2}$) & ($\times 10^{-2}$) & ($\times 10^{-2}$) & ($\times 10^{-3}$)"
        r" & ($\times 10^{-4}$) & ($\times 10^{-2}$) \\"
    )
    lines.append(r"        \midrule")

    classifier_items = list(by_classifier.items())
    for clf_i, (clf, calib_data_list) in enumerate(classifier_items):
        by_decon = collect_by_decon(calib_data_list)

        total_rows_for_clf = sum(len(v) for v in by_decon.values())
        lines.append(rf"        % --- {clf} ---")
        lines.append(rf"        \multirow{{{total_rows_for_clf}}}{{*}}{{{clf}}}")

        decon_items = list(by_decon.items())
        for dec_i, (decon, calib_rows) in enumerate(decon_items):
            n_calib = len(calib_rows)

            for row_j, (calib, vals) in enumerate(calib_rows):
                if row_j == 0 and n_calib > 1:
                    decon_cell = rf"\multirow{{{n_calib}}}{{*}}{{{decon}}}"
                elif row_j == 0 and n_calib == 1:
                    decon_cell = decon
                else:
                    decon_cell = ""

                row = (
                    f"                            & {decon_cell:<20} & {calib:<20} "
                    f"& {vals['R2']:<8} & {vals['LoA']:<25} & {vals['LoA (worst)']:<25} "
                    f"& {vals['MAE']:<10} & {vals['MSE']:<10} & {vals['KL']:<10} \\\\"
                )
                lines.append(row)

            is_last_decon = dec_i == len(decon_items) - 1
            is_last_clf = clf_i == len(classifier_items) - 1
            if not is_last_decon:
                lines.append(r"        \cmidrule(l){2-9}")
            elif not is_last_clf:
                lines.append(r"        \midrule")

    lines.append(r"        \bottomrule")
    lines.append(r"      \end{tabular}%")
    lines.append(r"    }}")
    lines.append(r"\end{table}")

    return "\n".join(lines)

In [29]:
custom_order = {"xgb": 0, "mlp": 1, "swn": 2, "nnls": 3, "psls": 4}

In [31]:
entries = [
    (
        "MethylBERT",
        "None",
        results_to_dataframe(results_test)
        .sort_values(by=["Deconvolver"], key=lambda x: x.map(custom_order))
        .to_dict(),
    ),
    (
        "MethylBERT",
        "Linear",
        results_to_dataframe(calibrated_test_results)
        .sort_values(by=["Deconvolver"], key=lambda x: x.map(custom_order))
        .to_dict(),
    ),
]

In [32]:
print(build_supp_table(entries))

\begin{table}[H]
  \centering
  \caption{Supplementary: Full results.}
  \label{tab:supp_results}
  {\footnotesize
    \resizebox{\textwidth}{!}{%
      \begin{tabular}{l cc cccc cc}
        \toprule
        \multirow{2}{*}{\textbf{Classifier}} & \multirow{2}{*}{\textbf{Deconvolver}} & \multirow{2}{*}{\textbf{Dec.\ Calib.}} & \textbf{$R^2$} & \textbf{LoA} & \textbf{LoA worst-class} & \textbf{MAE} & \textbf{MSE} & \textbf{KL} \\
        & &  & ($\times 10^{-2}$) & ($\times 10^{-2}$) & ($\times 10^{-2}$) & ($\times 10^{-3}$) & ($\times 10^{-4}$) & ($\times 10^{-2}$) \\
        \midrule
        % --- MethylBERT ---
        \multirow{10}{*}{MethylBERT}
                            & \multirow{2}{*}{XGB} & None                 & 94.29    & [-4.26, 4.26]             & [-10.48, 11.61]           & 5.439      & 4.73       & 11.50      \\
                            &                      & Linear               & 95.25    & [-3.43, 3.43]             & [-8.96, 9.25]             & 7.411      & 3.06